# RWPE Node Encoding — HI-Small

- **What:** random-walk positional encoding (RWPE) per account — step k = probability that a k-step random walk from the account is back at it (Dwivedi et al. 2021), on the directed transaction graph with P = D_out⁻¹ A: the maths of PyG `AddRandomWalkPE`.
- **Per snapshot, no leakage:** one file per snapshot from that snapshot's own edges (train / train+val / all). Self-loops (sender = receiver) are dropped when building P (section 3 shows why); the graph files stay unchanged.
- **Sizes:** k = 16 first; k = 8 (= steps 1–8) only by the rule in `EXPERIMENTS.md` §1. Output `Data/rwpe/rwpe_k{8,16}_{train,val,test}.pt`, float32 [515,070, k], rows in `account_to_idx` order.
- **Compute:** `rwpe_compute.py` (scipy, exact; logs non-zeros, seconds and RAM per step) as a subprocess — CPU only, locally or on Kaggle (`/kaggle/working/rwpe/` → the dataset `hi-small-rwpe`).

## 0. Setup

- Paths switch between the local repo (`Data/`, `Data/rwpe/`) and Kaggle (`hi-small-gnn` input, `/kaggle/working/rwpe/`).
- On Kaggle a plain `torch_geometric` install is enough (CPU, used only for the sanity checks) and the repo is cloned for `rwpe_compute.py`.

In [1]:
import gc
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch

IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    !pip install -q torch_geometric
    REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
    if not os.path.isdir(REPO_DIR):
        !git clone https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git {REPO_DIR}
    DATA_DIR, OUT_DIR = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn', '/kaggle/working/rwpe'
else:
    REPO_DIR, DATA_DIR, OUT_DIR = '.', 'Data', 'Data/rwpe'
sys.path.insert(0, REPO_DIR)

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.transforms import AddRandomWalkPE
import rwpe_compute as rc

os.makedirs(OUT_DIR, exist_ok=True)
SPLITS       = ('train', 'val', 'test')
WALK_LENGTHS = (8, 16)
N            = rc.NUM_NODES
print(f'torch {torch.__version__} | torch_geometric {torch_geometric.__version__} | {DATA_DIR} -> {OUT_DIR}')

torch 2.8.0+cpu | torch_geometric 2.6.1 | Data -> Data/rwpe


## 1. Sanity test on a toy graph

- 4 accounts, edges A→B, B→A, A→C, C→D, D→A; the k = 8 vectors of A and B were worked out by hand (walks follow the edge direction, each step divides by the out-degree).
- Checks PyG `AddRandomWalkPE` against them, and `rwpe_compute.compute_rwpe` against PyG.

In [2]:
A, B, C, D = 0, 1, 2, 3
toy_edges = torch.tensor([[A, B, A, C, D],
                          [B, A, C, D, A]])             # A->B, B->A, A->C, C->D, D->A
expected = {A: [0, 0.5, 0.5, 0.25, 0.5, 0.375, 0.375, 0.4375],
            B: [0, 0.5, 0, 0.25, 0.25, 0.125, 0.25, 0.1875]}

pyg  = AddRandomWalkPE(walk_length=8, attr_name='rwpe')(Data(edge_index=toy_edges, num_nodes=4)).rwpe
ours = torch.from_numpy(rc.compute_rwpe(toy_edges[0].numpy(), toy_edges[1].numpy(), 4, 8, log=lambda line: None))
print('PyG :\n', pyg.numpy().round(4))
print('ours:\n', ours.numpy().round(4))
for node, values in expected.items():
    assert torch.allclose(pyg[node],  torch.tensor(values)), f'PyG differs from the hand calculation at node {node}'
    assert torch.allclose(ours[node], torch.tensor(values)), f'rwpe_compute differs from the hand calculation at node {node}'
assert torch.allclose(pyg, ours, atol=1e-6)
print('OK: walks follow edge direction, steps divide by out-degree, rwpe_compute == AddRandomWalkPE')

PyG :
 [[0.     0.5    0.5    0.25   0.5    0.375  0.375  0.4375]
 [0.     0.5    0.     0.25   0.25   0.125  0.25   0.1875]
 [0.     0.     0.5    0.     0.25   0.25   0.125  0.25  ]
 [0.     0.     0.5    0.     0.25   0.25   0.125  0.25  ]]
ours:
 [[0.     0.5    0.5    0.25   0.5    0.375  0.375  0.4375]
 [0.     0.5    0.     0.25   0.25   0.125  0.25   0.1875]
 [0.     0.     0.5    0.     0.25   0.25   0.125  0.25  ]
 [0.     0.     0.5    0.     0.25   0.25   0.125  0.25  ]]
OK: walks follow edge direction, steps divide by out-degree, rwpe_compute == AddRandomWalkPE


## 2. Edge lists and the check against the graph snapshots

- Edge list = `src/dst_account` of `Data/edge_features.csv` mapped through `account_to_idx.pkl`, exactly how `Data_preparation.ipynb` built `edge_index`; cached as `edge_index_all.npy` for the script.
- Each cumulative snapshot = the first n rows; asserted equal to the `.pt` file's `edge_index` and labels, one graph at a time (1–2 GB each).

In [3]:
edge_index = rc.load_edge_index(DATA_DIR, OUT_DIR)                    # [2, 5,077,237] int32
meta       = json.load(open(f'{DATA_DIR}/feature_meta.json'))
pay_cols   = [c for c in meta['BASE_EDGE_COLS'] if c.startswith('PayFmt_')]
edges      = pd.read_csv(f'{DATA_DIR}/edge_features.csv', usecols=['label', 'Is_Self_Loop'] + pay_cols)
label      = edges['label'].to_numpy()
assert edge_index.shape == (2, len(edges)) and len(edges) == rc.SPLIT_EDGES['test']
print(f'edge_index {edge_index.shape} | nodes {N:,} | laundering edges {int(label.sum()):,}')

starts = {'train': 0, 'val': rc.SPLIT_EDGES['train'], 'test': rc.SPLIT_EDGES['val']}   # evaluated window of each split
for split in SPLITS:
    graph = torch.load(f'{DATA_DIR}/{split}_graph.pt', weights_only=False)
    n = rc.SPLIT_EDGES[split]
    assert graph.num_nodes == N and graph.edge_index.shape[1] == n
    assert np.array_equal(graph.edge_index.numpy(), edge_index[:, :n]), f'{split}: edge_index differs from CSV + account_to_idx'
    eval_rows = graph.eval_mask.numpy().nonzero()[0]
    assert eval_rows[0] == starts[split] and eval_rows[-1] == n - 1
    assert np.array_equal(graph.y.numpy()[eval_rows], label[eval_rows]), f'{split}: labels differ'
    print(f'{split:<5} graph.edge_index == CSV edge list ({n:,} edges) | evaluated rows {starts[split]:,}..{n - 1:,} | labels match')
    del graph
    gc.collect()

edge_index (2, 5077237) | nodes 515,070 | laundering edges 4,522


train graph.edge_index == CSV edge list (3,046,342 edges) | evaluated rows 0..3,046,341 | labels match


val   graph.edge_index == CSV edge list (4,061,789 edges) | evaluated rows 3,046,342..4,061,788 | labels match


test  graph.edge_index == CSV edge list (5,077,237 edges) | evaluated rows 4,061,789..5,077,236 | labels match


## 3. Self-loop check

- Why first: the decision to drop self-loops from P (`CLAUDE.md` §3b) rests on these numbers — share per split, which payment formats they are, how many accounts have one and whether those accounts launder more.
- Report only; the graph and the model inputs keep every self-loop.

In [4]:
is_loop = edge_index[0] == edge_index[1]
assert np.array_equal(is_loop, edges['Is_Self_Loop'].to_numpy().astype(bool))

rows = []
for split in SPLITS:
    a, b = starts[split], rc.SPLIT_EDGES[split]
    loops, y = is_loop[a:b], label[a:b]
    rows.append({'split': split, 'edges': b - a, 'self_loops': int(loops.sum()), 'share': loops.mean(),
                 'laundering_rate_self_loop': y[loops].mean(), 'laundering_rate_other': y[~loops].mean()})
rows.append({'split': 'all', 'edges': len(is_loop), 'self_loops': int(is_loop.sum()), 'share': is_loop.mean(),
             'laundering_rate_self_loop': label[is_loop].mean(), 'laundering_rate_other': label[~is_loop].mean()})
display(pd.DataFrame(rows).round(5))

pay_format = np.array([c.replace('PayFmt_', '') for c in pay_cols])[edges[pay_cols].to_numpy().argmax(1)]
by_format  = pd.crosstab(pay_format, is_loop, margins=True)
by_format.columns = ['other', 'self_loop', 'all']
by_format['self_loop_share'] = (by_format['self_loop'] / by_format['all']).round(3)
display(by_format)

loop_accounts = np.zeros(N, dtype=bool)
loop_accounts[edge_index[0][is_loop]] = True
touches = loop_accounts[edge_index[0]] | loop_accounts[edge_index[1]]
print(f'accounts with >= 1 self-loop: {loop_accounts.sum():,} ({loop_accounts.mean():.2%})')
print(f'laundering rate of edges touching such an account: {label[touches].mean():.4%} vs {label[~touches].mean():.4%} for the rest')

,split,edges,self_loops,share,laundering_rate_self_loop,laundering_rate_other
0,train,3046342,547862,0.17984,0.00001,0.00092
1,val,1015447,21896,0.02156,0.00005,0.00109
2,test,1015448,21061,0.02074,0.00009,0.00115
3,all,5077237,590819,0.11637,0.00001,0.00101


,other,self_loop,all,self_loop_share
row_0,,,,
ACH,528735,70954,599689,0.118
Bitcoin,119774,26317,146091,0.180
Cash,489436,1455,490891,0.003
Cheque,1858265,6066,1864331,0.003
Credit Card,1319148,4176,1323324,0.003
Reinvestment,0,481056,481056,1.000
Wire,171060,795,171855,0.005
All,4486418,590819,5077237,0.116


accounts with >= 1 self-loop: 367,753 (71.40%)
laundering rate of edges touching such an account: 0.0902% vs 0.0626% for the rest


## 4. Compute RWPE

- `rwpe_compute.py <split> <k>` as a subprocess per snapshot and k (long jobs stay out of the notebook); train k = 8 first, the measurement run.
- Files that already exist are skipped; every job's per-step log (non-zeros of Pᵏ, seconds, RAM) is in `log_<split>_k<k>.txt` and summarised below.

In [5]:
def compute(split, k):
    out_path = f'{OUT_DIR}/rwpe_k{k}_{split}.pt'
    if os.path.exists(out_path):
        print(f'{out_path} exists, skipped')
        return
    cmd = [sys.executable, f'{REPO_DIR}/rwpe_compute.py', split, str(k), DATA_DIR, OUT_DIR]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    assert result.returncode == 0, result.stderr[-2000:]

compute('train', 8)                      # measurement first
for k in WALK_LENGTHS:
    for split in SPLITS:
        compute(split, k)

Data/rwpe/rwpe_k8_train.pt exists, skipped
Data/rwpe/rwpe_k8_train.pt exists, skipped
Data/rwpe/rwpe_k8_val.pt exists, skipped
Data/rwpe/rwpe_k8_test.pt exists, skipped
Data/rwpe/rwpe_k16_train.pt exists, skipped
Data/rwpe/rwpe_k16_val.pt exists, skipped
Data/rwpe/rwpe_k16_test.pt exists, skipped


In [6]:
summary = []
for k in WALK_LENGTHS:
    for split in SPLITS:
        lines = open(f'{OUT_DIR}/log_{split}_k{k}.txt').read().splitlines()
        last_step, total = [l for l in lines if l.startswith('step')][-1], lines[-1]
        summary.append({'split': split, 'k': k,
                        'nnz(P^k)':    int(last_step.split('=')[1].split('|')[0].replace(',', '')),
                        'total_s':     int(total.split('total ')[1].split('s')[0]),
                        'peak_RAM_GB': float(total.split('peak RAM ')[1].split(' GB')[0])})
display(pd.DataFrame(summary))
print(open(f'{OUT_DIR}/log_test_k16.txt').read())      # the largest job, step by step

,split,k,nnz(P^k),total_s,peak_RAM_GB
0,train,8,45570372,31,0.97
1,val,8,52496772,11,1.04
2,test,8,59907185,12,1.14
3,train,16,99792575,47,1.76
4,val,16,144181203,59,2.29
5,test,16,205425596,82,2.89


test: 5,077,237 edges, 590,819 self-loops (dropped), K=16, nodes=515,070
step  1 | nnz(P^1)=      647,316 | peak RAM 0.35 GB
step  2 | nnz(P^2)=   12,281,103 |    0.2s | RAM now 0.43 GB | peak 0.44 GB
step  3 | nnz(P^3)=   16,408,045 |    0.6s | RAM now 0.46 GB | peak 0.56 GB
step  4 | nnz(P^4)=   22,784,751 |    1.0s | RAM now 0.51 GB | peak 0.64 GB
step  5 | nnz(P^5)=   30,216,122 |    1.5s | RAM now 0.56 GB | peak 0.74 GB
step  6 | nnz(P^6)=   38,965,794 |    2.0s | RAM now 0.63 GB | peak 0.86 GB
step  7 | nnz(P^7)=   48,712,341 |    2.6s | RAM now 0.70 GB | peak 1.00 GB
step  8 | nnz(P^8)=   59,907,185 |    3.4s | RAM now 0.79 GB | peak 1.15 GB
step  9 | nnz(P^9)=   71,917,790 |    4.2s | RAM now 0.88 GB | peak 1.33 GB
step 10 | nnz(P^10)=   85,082,409 |    5.0s | RAM now 0.97 GB | peak 1.51 GB
step 11 | nnz(P^11)=   99,274,762 |    6.5s | RAM now 1.08 GB | peak 1.72 GB
step 12 | nnz(P^12)=  114,827,361 |    8.6s | RAM now 1.19 GB | peak 1.94 GB
step 13 | nnz(P^13)=  132,367,345 | 

## 5. Checks

- Every file: shape [515,070, k], float32, values in [0, 1], no NaN; step 1 is 0 for every account (self-loops removed); `k16[:, :8] == k8` exactly.
- The script against PyG `AddRandomWalkPE` on the real train snapshot (same self-loop-free edges), k = 8.

In [7]:
rwpe = {(split, k): torch.load(f'{OUT_DIR}/rwpe_k{k}_{split}.pt', weights_only=True)
        for k in WALK_LENGTHS for split in SPLITS}
for (split, k), x in rwpe.items():
    assert x.shape == (N, k) and x.dtype == torch.float32, (split, k, x.shape, x.dtype)
    assert not torch.isnan(x).any() and x.min() >= 0 and x.max() <= 1
    assert (x[:, 0] == 0).all(), f'{split} k={k}: step 1 must be 0 without self-loops'
for split in SPLITS:
    assert torch.equal(rwpe[(split, 16)][:, :8], rwpe[(split, 8)]), f'{split}: k16[:, :8] != k8'
print('OK: shapes [515,070, k], float32, values in [0, 1], no NaN, step 1 == 0, k16[:, :8] == k8')

src, dst = edge_index[:, :rc.SPLIT_EDGES['train']]
keep = src != dst
t0 = time.time()
pyg_train = AddRandomWalkPE(walk_length=8, attr_name='rwpe')(
    Data(edge_index=torch.from_numpy(np.stack([src[keep], dst[keep]]).astype(np.int64)), num_nodes=N)).rwpe
print(f'PyG AddRandomWalkPE on the train snapshot, k = 8: {time.time() - t0:.0f}s')
assert torch.allclose(pyg_train, rwpe[('train', 8)], atol=1e-6)
print(f'OK: rwpe_compute == AddRandomWalkPE on train (max |diff| {(pyg_train - rwpe[("train", 8)]).abs().max():.1e})')
del pyg_train
gc.collect()

OK: shapes [515,070, k], float32, values in [0, 1], no NaN, step 1 == 0, k16[:, :8] == k8


C:\Users\User\.conda\envs\graph_feature_preprocessor\lib\site-packages\torch_geometric\transforms\add_positional_encoding.py:169: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  out = out @ adj


PyG AddRandomWalkPE on the train snapshot, k = 8: 29s
OK: rwpe_compute == AddRandomWalkPE on train (max |diff| 4.8e-07)


16

## 6. Diagnostics (report only)

- Per snapshot: accounts that only receive (out-degree 0) with and without self-loops; all-zero RWPE rows; accounts with a self-loop; laundering edges whose sender or receiver has an all-zero RWPE.
- The "with self-loops" all-zero share comes from one extra k = 16 run on train that keeps them (temporary file, deleted).
- Then: how rare a non-zero RWPE is, and whether such accounts are enriched among laundering edges.

In [8]:
rows = []
for split in SPLITS:
    src, dst = edge_index[:, :rc.SPLIT_EDGES[split]]
    loops = src == dst
    out_deg_with, out_deg_without = np.bincount(src, minlength=N), np.bincount(src[~loops], minlength=N)
    all_zero = ~(rwpe[(split, 16)].numpy() > 0).any(1)
    eval_rows = np.arange(starts[split], rc.SPLIT_EDGES[split])
    laundering = eval_rows[label[eval_rows] == 1]
    rows.append({'split': split,
                 'only_receive_with_loops':    (out_deg_with == 0).mean(),
                 'only_receive_without_loops': (out_deg_without == 0).mean(),
                 'self_loop_accounts':         len(np.unique(src[loops])) / N,
                 'all_zero_rwpe16':            all_zero.mean(),
                 'laundering_edges_with_all_zero_endpoint': (all_zero[src[laundering]] | all_zero[dst[laundering]]).mean()})
display(pd.DataFrame(rows).round(4))

# train k = 16 with self-loops kept, for the before/after comparison only
tmp_dir = f'{OUT_DIR}/_withloops'
os.makedirs(tmp_dir, exist_ok=True)
shutil.copy(f'{OUT_DIR}/edge_index_all.npy', tmp_dir)
result = subprocess.run([sys.executable, f'{REPO_DIR}/rwpe_compute.py', 'train', '16', DATA_DIR, tmp_dir, '--keep-self-loops'],
                        capture_output=True, text=True)
assert result.returncode == 0, result.stderr[-2000:]
with_loops = torch.load(f'{tmp_dir}/rwpe_k16_train_withloops.pt', weights_only=True)
print(f'train, all-zero RWPE rows: with self-loops {(~(with_loops > 0).any(1)).float().mean():.2%} -> '
      f'without {(~(rwpe[("train", 16)] > 0).any(1)).float().mean():.2%}')
print(f'train, step 1 with self-loops: non-zero for {(with_loops[:, 0] > 0).float().mean():.2%} of accounts (mean {with_loops[:, 0].mean():.4f})')
shutil.rmtree(tmp_dir)

mean_per_step = pd.DataFrame({split: rwpe[(split, 16)].mean(0).numpy() for split in SPLITS}, index=range(1, 17))
mean_per_step.index.name = 'step'
display(mean_per_step.round(5))

# how rare is a non-zero RWPE, and are such accounts enriched among laundering edges?
for split in SPLITS:
    src, dst = edge_index[:, :rc.SPLIT_EDGES[split]]
    nonzero = (rwpe[(split, 16)].numpy() > 0).any(1)
    eval_rows = np.arange(starts[split], rc.SPLIT_EDGES[split])
    laundering = eval_rows[label[eval_rows] == 1]
    either = nonzero[src] | nonzero[dst]
    print(f'{split:<5} accounts with a non-zero RWPE {nonzero.mean():.2%} ({nonzero.sum():,}) | '
          f'evaluated edges with a non-zero endpoint {either[eval_rows].mean():.2%}, laundering edges {either[laundering].mean():.2%} | '
          f'laundering rate with / without such an endpoint {label[eval_rows][either[eval_rows]].mean():.3%} / {label[eval_rows][~either[eval_rows]].mean():.3%}')


,split,only_receive_with_loops,only_receive_without_loops,self_loop_accounts,all_zero_rwpe16,laundering_edges_with_all_zero_endpoint
0,train,0.0432,0.4579,0.7128,0.9942,0.8938
1,val,0.0392,0.4324,0.7134,0.9898,0.8863
2,test,0.0351,0.4067,0.7140,0.9837,0.8714


train, all-zero RWPE rows: with self-loops 28.62% -> without 99.42%
train, step 1 with self-loops: non-zero for 71.28% of accounts (mean 0.4978)


,train,val,test
step,,,
1,0.00000,0.00000,0.00000
2,0.00255,0.00435,0.00678
3,0.00006,0.00011,0.00022
4,0.00230,0.00393,0.00613
5,0.00006,0.00012,0.00024
6,0.00222,0.00381,0.00594
7,0.00006,0.00012,0.00024
8,0.00219,0.00376,0.00586
9,0.00006,0.00012,0.00024


train accounts with a non-zero RWPE 0.58% (2,990) | evaluated edges with a non-zero endpoint 2.05%, laundering edges 15.19% | laundering rate with / without such an endpoint 0.559% / 0.065%
val   accounts with a non-zero RWPE 1.02% (5,264) | evaluated edges with a non-zero endpoint 3.72%, laundering edges 16.82% | laundering rate with / without such an endpoint 0.482% / 0.092%


test  accounts with a non-zero RWPE 1.63% (8,414) | evaluated edges with a non-zero endpoint 5.31%, laundering edges 22.40% | laundering rate with / without such an endpoint 0.475% / 0.092%


## 7. Value scale

- Distribution per step on the train snapshot (k = 16): share of zeros and quantiles of the non-zero values.
- Decides whether a scaler (fit on train only) is needed before `node_proj`; proposal below, not applied here.

In [9]:
x = rwpe[('train', 16)].numpy()
scale = []
for step in range(16):
    v = x[:, step]
    nz = v[v > 0]
    scale.append({'step': step + 1, 'share_zero': (v == 0).mean(), 'mean': v.mean(),
                  'p50_nonzero': np.median(nz) if len(nz) else 0, 'p90_nonzero': np.quantile(nz, 0.9) if len(nz) else 0,
                  'p99_nonzero': np.quantile(nz, 0.99) if len(nz) else 0, 'max': v.max()})
display(pd.DataFrame(scale).set_index('step').round(4))
print('entity one-hot node features: values in {0, 1}, mean per column', np.round(torch.load(f'{DATA_DIR}/train_graph.pt', weights_only=False).x.mean(0).numpy(), 3))

,share_zero,mean,p50_nonzero,p90_nonzero,p99_nonzero,max
step,,,,,,
1,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,0.9954,0.0026,0.5000,1.0000,1.0000,1.0000
3,0.9997,0.0001,0.2222,0.2500,1.0000,1.0000
4,0.9954,0.0023,0.2500,1.0000,1.0000,1.0000
5,0.9997,0.0001,0.1826,0.3125,0.3938,0.4444
6,0.9953,0.0022,0.1250,1.0000,1.0000,1.0000
7,0.9996,0.0001,0.1000,0.3281,0.4196,0.4444
8,0.9953,0.0022,0.0736,1.0000,1.0000,1.0000
9,0.9995,0.0001,0.0086,0.3320,0.7244,1.0000


entity one-hot node features: values in {0, 1}, mean per column [0.333 0.013 0.    0.001 0.366 0.287]


**Proposal (not applied): no scaler.** RWPE already lives in [0, 1] like the entity one-hots next to it, and every column is > 99 % zeros: standardising on train would turn the zeros into a constant offset and the rare informative values into outliers (~15 σ), which only hurts a linear `node_proj`. Values are stored as they are; `NODE_ENC = rwpe16` appends them unchanged.

**What the files say:** without self-loops, money mostly flows forward — only ~0.6 % of accounts have any closed directed walk of length ≤ 16, so RWPE is all-zero for the rest (table above). Those few accounts are the ones on directed cycles, and they are much more common at the ends of laundering edges than of ordinary edges (last cell of section 6). Report only: the walk direction and the graph are unchanged, as decided.